In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [2]:
df_first_mapping = pd.read_csv(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/mapping.csv",
    sep=";"
)
code_map_1 = dict(
    zip(df_first_mapping["gtap"], df_first_mapping["iso3c"])
)

In [3]:
df_country_name_mapping = pd.read_excel(
    Path(
        "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/Aggregation111Power_trump.xlsx"
    ),
    sheet_name="Mapping_table_country",
    header=1,
).drop(columns=["Label"])


One row is missing in the matching table :

"Other Asia, nes" : S19

So I add it manually in RoW. According to IA, it could be Taiwan, so it may be corrected by classifying S19 to XTW.

In [4]:
additionnal_row = pd.DataFrame([{"Code": "S19", "Aggregation Code": "RoW"}])
df_country_name_mapping = pd.concat([df_country_name_mapping, additionnal_row], ignore_index=True)
code_map_2 = dict(
    zip(df_country_name_mapping["Code"], df_country_name_mapping["Aggregation Code"])
)


This df is created in the notebook `data_formatting.ipynb`

In [5]:
df_MIRAGE = pd.read_parquet(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/MIRAGE_grouped_data.parquet"
).drop(columns="Quantity")

In [6]:
df_debug = pd.merge(
    left=df_MIRAGE,
    right=df_first_mapping,
    left_on="Exporter",
    right_on="gtap",
    how="left",
)
# df_debug = df_debug.drop(columns=["Exporter", "gtap"]).rename(columns={"iso3c": "Exporter"})[df_debug["iso3c"].isna()]["Exporter"].unique()


In [7]:
df_country_name_mapping.head()

,Code,Aggregation Code
0,ARG,RoLAC
1,BGD,RoW
2,IDN,ASEAN
3,LKA,RoW
4,MNG,RoW


In [8]:
df_debug


,Year,Exporter,Importer,MIRAGE_code,k$,iso3c,gtap
0,2017,ABW,AUT,Electrical,0.160,XCB,ABW
1,2017,ABW,AUT,Electronic,64.057,XCB,ABW
2,2017,ABW,AUT,Vehicles,0.285,XCB,ABW
3,2017,ABW,AUT,OthManuf,44.876,XCB,ABW
4,2017,ABW,BEL,Electrical,7.547,XCB,ABW
...,...,...,...,...,...,...,...
3729333,2024,ZWE,ZMB,Cereals,22.748,ZWE,ZWE
3729334,2024,ZWE,ZMB,AnimAgri,0.002,ZWE,ZWE
3729335,2024,ZWE,ZMB,Machinery,0.043,ZWE,ZWE
3729336,2024,ZWE,ZMB,Mineral,1731.311,ZWE,ZWE


In [9]:
df_MIRAGE["Exporter"] = df_MIRAGE["Exporter"].map(code_map_1).fillna("S19") 
# I added the S19 matching in the second mapping, so I must fillna after the first one
df_MIRAGE["Importer"] = df_MIRAGE["Importer"].map(code_map_1).fillna("S19")

In [10]:
df_MIRAGE["Exporter"] = df_MIRAGE["Exporter"].map(code_map_2)
df_MIRAGE["Importer"] = df_MIRAGE["Importer"].map(code_map_2)


## Market share

In [11]:
df_grouped = df_MIRAGE.groupby(["Year", "MIRAGE_code"])["k$"].sum().reset_index()

df_all = pd.merge(
    left=df_MIRAGE,
    right=df_grouped,
    on=["Year", "MIRAGE_code"],
    how="left",
    suffixes=("", "_total"),
)

df_all["market_share"] = df_all["k$"] / df_all["k$_total"]

df_all = (
    df_all.copy()
    .groupby(["Year", "MIRAGE_code", "Exporter"])
    .agg({"k$": "sum", "market_share": "sum"})
    .reset_index()
)
df_all.sort_values(["Year", "Exporter", "MIRAGE_code"], inplace=True)
df_all


,Year,MIRAGE_code,Exporter,k$,market_share
0,2017,AnimAgri,ASEAN,1.348695e+06,0.028990
22,2017,BevTabacco,ASEAN,6.018020e+06,0.038788
44,2017,Cereals,ASEAN,6.305813e+05,0.007509
66,2017,Chemistry,ASEAN,9.612210e+07,0.073986
88,2017,Coal,ASEAN,2.514624e+07,0.187442
...,...,...,...,...,...
4827,2024,RefinedOil,USA,4.641399e+07,0.280345
4849,2024,Ruplastic,USA,4.793285e+07,0.079745
4871,2024,Textile,USA,2.085936e+07,0.018758
4893,2024,Vehicles,USA,1.520590e+08,0.080910


In [12]:
df_all.to_csv(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/MIRAGE_grouped_data_cleaned.csv",
    index=False,
)


## Market share progression

In [13]:
df_MIRAGE["Year"].unique()

array([2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024])

In [14]:
# Regroupement des montant par secteur
agg = df_MIRAGE.groupby(["Exporter", "MIRAGE_code", "Year"], as_index=False)["k$"].sum()

# Rotation pour avoir une colonne par année
wide = agg.pivot(index=["Exporter", "MIRAGE_code"], columns="Year", values="k$")

years = list(range(2019, 2025))
result = pd.DataFrame(index=wide.index)
for y1, y2 in zip(years[:-1], years[1:]):
    result[f"évolution_{y1}_{y2}"] = wide[y2] / wide[y1] - 1 # ratio année N+1 et année N
result["évolution_2019_2024"] = wide[2024] / wide[2019] - 1 # ratio 2024 / 2019

result = result.reset_index().rename(columns={"Exporter": "Pays"})

In [15]:
result.to_csv("/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/market_share_evolution.csv", index=False)

### Analysis

In [16]:
agg_all = df_MIRAGE.groupby(["MIRAGE_code", "Year"], as_index=False)["k$"].sum()

# Rotation pour avoir une colonne par année
wide_all = agg_all.pivot(index=["MIRAGE_code"], columns="Year", values="k$")

years = list(range(2019, 2025))
result_all = pd.DataFrame(index=wide_all.index)
for y1, y2 in zip(years[:-1], years[1:]):
    result_all[f"évolution_{y1}_{y2}"] = (
        wide_all[y2] / wide_all[y1] - 1
    )  # ratio année N+1 et année N
result_all["évolution_2019_2024"] = wide_all[2024] / wide_all[2019] - 1  # ratio 2024 / 2019

result_all = result_all.reset_index().rename(columns={"Exporter": "Pays"})
result_all[["MIRAGE_code", "évolution_2019_2024"]].sort_values(
    by="évolution_2019_2024", ascending=False
)

,MIRAGE_code,évolution_2019_2024
22,Power,0.761893
9,Gas,0.533061
23,RefinedOil,0.459569
18,OthFood,0.386680
21,Pharmacy,0.364770
16,OilSeeds,0.363563
10,MNonFer,0.344893
4,Coal,0.338103
6,Electrical,0.336448
17,OthCrops,0.289181
